### nb_03 — オントロジーの構造とメタデータをコードで構築する

**アタッチするレイクハウス**：`lh_its_asset_silver`

#### 前提

Step 2 で、UI から次を作成・設定済みであること。

**エンティティ型5個**

| 名前 | シルバー層テーブル | キー |
|---|---|---|
| `Person` | `sv_person` | `PersonId` |
| `Project` | `sv_project` | `ProjectId` |
| `Customer` | `sv_customer` | `CustomerId` |
| `Organization` | `sv_organization` | `OrganizationId` |
| `Assignment` | `sv_assignment` | `AssignmentKey`（生成キー） |

**関係型3個**

| 名前 | 起点 → 終点 | 結合 |
|---|---|---|
| `belongsTo` | Person → Organization | `sv_person` の `OrganizationId` |
| `performedBy` | Assignment → Person | `sv_assignment` の `PersonId` |
| `deliveredFor` | Project → Customer | `sv_project` の `CustomerId` |

**メタデータ（セマンティックエンリッチメント）**

- エンティティ型 `Person` に、説明・シノニム・追加メタデータを各1件
- 関係型 `belongsTo` に、説明を1件

手で作る意味は「オントロジーの構成要素を体で覚えること」です。この5個+3個で、
通常のエンティティ型・関連実体と生成キー・エンティティ自身のテーブルで結ぶ関係・
関連実体から結ぶ関係を、すべて一度は経験できます。これ以上は同じ操作の繰り返しなので、
残りはこのノートブックで作ります。

#### このノートブックがすること

| フェーズ | 内容 |
|---|---|
| 1. 構造をつくる | 既存の定義を取得し、まだ無いエンティティ型12個・関係型16個を追加する |
| 2. 意味を与える | 各定義の `semanticEnrichment` に説明・シノニム・追加メタデータを書き込む |
| 3. 反映する | Update Item Definition API で一度に定義を更新する |
| 4. グラフを構築する | オントロジーの定義からグラフを生成し、データを取り込む |

**何度実行しても同じ結果になります**（ID は名前から決定論的に生成）。手動で設定した
内容は上書きしません。

メタデータは各 `definition.json` の `semanticEnrichment` に格納されます。公式スキーマには
記載がありませんが、Fabric はこの形で受け付けます。

#### 準備

In [ ]:
import base64
import json
import time
import uuid

import requests
import notebookutils

#### 設定

`SILVER_SCHEMA` は環境に合わせてください。スキーマ無効レイクハウスなら `None` です。

In [ ]:
# ============================================================
# 設定
# ============================================================
ONTOLOGY_NAME = "ont_its_asset"        # Step 2 で作成したオントロジーアイテム名
SILVER_LAKEHOUSE_NAME = "lh_its_asset_silver"
SILVER_SCHEMA = "dbo"                    # スキーマ無効レイクハウスなら None

# メタデータ（セマンティックエンリッチメント）の適用
APPLY_METADATA = True

# 手動設定済みのメタデータを上書きするか。
# False なら、UI で設定した内容はそのまま残す（Step 2 の手動設定分を保護する）。
OVERWRITE_EXISTING_METADATA = False

FABRIC_API = "https://api.fabric.microsoft.com/v1"

# 実行中のワークスペース ID を取得
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]

token = notebookutils.credentials.getToken("pbi")
HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

#### エンティティ型の定義

ここに並んでいるのは**バインドするプロパティだけ**です。`_valid_as_of` やソース由来の
ID（`PersonSkillId` など）は意図的に含めていません。理由は `docs/property-binding.md`
を参照してください。

In [ ]:
# ============================================================
# 作成するエンティティ型の定義
# ============================================================
# name: (シルバー層テーブル, エンティティ型キー, [(プロパティ名, 型), ...])
#
# 型は String / BigInt / Double / DateTime / Boolean / Object から選ぶ。
# 日付は String のままにしている（CSV 由来で型が揺れるため。実案件では DateTime を推奨）。
#
# ここに並んでいるのは「バインドするプロパティ」だけです。
# _valid_as_of やソース由来の ID（PersonSkillId など）は意図的に含めていません。
# 理由は docs/property-binding.md を参照してください。

S, N, D = "String", "BigInt", "Double"

ENTITY_TYPES = {
    # --- 手動で作成済みの5個（既存なら触らない。参照のためだけに記載）---
    "Person": ("sv_person", "PersonId", [
        ("PersonId", S), ("FullName", S), ("OrganizationId", S), ("OrganizationName", S),
        ("JobTitle", S), ("YearsOfService", N), ("WorkLocation", S),
        ("EmploymentType", S), ("Email", S)]),
    "Project": ("sv_project", "ProjectId", [
        ("ProjectId", S), ("ProjectName", S), ("CustomerId", S), ("SolutionDomain", S),
        ("ContractType", S), ("StartDate", S), ("EndDate", S), ("Status", S),
        ("RevenueJPY", N), ("CostJPY", N), ("MarginRate", D),
        ("DevelopmentMethod", S), ("PlatformType", S)]),
    "Customer": ("sv_customer", "CustomerId", [
        ("CustomerId", S), ("CustomerName", S), ("Industry", S), ("Region", S),
        ("AccountTier", S), ("PrimaryBusinessChallenge", S), ("AnnualITBudgetOkuJPY", N)]),

    "Organization": ("sv_organization", "OrganizationId", [
        ("OrganizationId", S), ("OrganizationName", S), ("DomainFocus", S),
        ("OrganizationType", S)]),
    "Assignment": ("sv_assignment", "AssignmentKey", [
        ("AssignmentKey", S), ("PersonId", S), ("ProjectId", S), ("ProjectRole", S),
        ("StartDate", S), ("EndDate", S), ("AllocationPercent", N),
        ("ContributionResult", S)]),

    # --- ここから下がコードで作成される12個 ---
    "Skill": ("sv_skill", "SkillId", [
        ("SkillId", S), ("SkillName", S), ("SkillCategory", S), ("SkillType", S)]),
    "Certification": ("sv_certification", "CertificationId", [
        ("CertificationId", S), ("CertificationName", S), ("Issuer", S), ("Domain", S)]),
    "Technology": ("sv_technology", "TechnologyId", [
        ("TechnologyId", S), ("TechnologyName", S), ("Vendor", S), ("TechnologyCategory", S)]),
    "Stakeholder": ("sv_stakeholder", "StakeholderId", [
        ("StakeholderId", S), ("CustomerId", S), ("StakeholderName", S),
        ("StakeholderRole", S), ("InfluenceLevel", S), ("RelationshipStatus", S)]),
    "Opportunity": ("sv_opportunity", "OpportunityId", [
        ("OpportunityId", S), ("CustomerId", S), ("OpportunityName", S), ("SalesStage", S),
        ("ExpectedRevenueJPY", N), ("ExpectedCloseDate", S),
        ("RequiredSkillIds", S), ("OwnerPersonId", S)]),
    "Deliverable": ("sv_deliverable", "DeliverableId", [
        ("DeliverableId", S), ("ProjectId", S), ("Title", S), ("DocumentType", S),
        ("Confidentiality", S), ("DocumentUrl", S), ("CreatedDate", S),
        ("ReusabilityRating", S)]),
    "Knowledge": ("sv_knowledge", "KnowledgeId", [
        ("KnowledgeId", S), ("ProjectId", S), ("KnowledgeType", S), ("Title", S),
        ("Summary", S), ("Tags", S), ("AuthorPersonId", S), ("RegisteredDate", S),
        ("ReuseValue", S)]),
    # 関連実体。キーは nb_02 が業務キーから生成した *Key 列
    "Availability": ("sv_availability", "AvailabilityKey", [
        ("AvailabilityKey", S), ("PersonId", S), ("YearMonth", S),
        ("AllocatedPercent", N), ("AvailablePercent", N), ("AvailabilityStatus", S),
        ("DataAsOfDate", S)]),
    "PersonSkill": ("sv_person_skill", "PersonSkillKey", [
        ("PersonSkillKey", S), ("PersonId", S), ("SkillId", S), ("ProficiencyLevel", S),
        ("YearsOfExperience", N), ("EvidenceProjectId", S), ("EvidenceType", S)]),
    "PersonCertification": ("sv_person_certification", "PersonCertificationKey", [
        ("PersonCertificationKey", S), ("PersonId", S), ("CertificationId", S),
        ("AcquiredDate", S), ("ExpiryDate", S)]),
    "ProjectTechnology": ("sv_project_technology", "ProjectTechnologyKey", [
        ("ProjectTechnologyKey", S), ("ProjectId", S), ("TechnologyId", S), ("UsageType", S)]),
    "ProjectRequiredSkill": ("sv_project_required_skill", "ProjectRequiredSkillKey", [
        ("ProjectRequiredSkillKey", S), ("ProjectId", S), ("SkillId", S), ("Requirement", S)]),
}

# 表示名に使うプロパティ（省略時はキー）
DISPLAY_NAME_PROPERTY = {
    "Person": "FullName", "Project": "ProjectName", "Customer": "CustomerName",
    "Organization": "OrganizationName", "Skill": "SkillName",
    "Certification": "CertificationName", "Technology": "TechnologyName",
    "Stakeholder": "StakeholderName", "Opportunity": "OpportunityName",
    "Deliverable": "Title", "Knowledge": "Title",
}

#### 関係型の定義

In [ ]:
# ============================================================
# 作成する関係型の定義
# ============================================================
# (関係型名, 起点エンティティ型, 終点エンティティ型,
#  マッピングテーブル, 起点の突き合わせ列, 終点の突き合わせ列)
RELATIONSHIP_TYPES = [
    ("belongsTo",             "Person",               "Organization", "sv_person",                 "PersonId",              "OrganizationId"),
    ("performedBy",           "Assignment",           "Person",       "sv_assignment",             "AssignmentKey",         "PersonId"),
    ("assignedTo",            "Assignment",           "Project",      "sv_assignment",             "AssignmentKey",         "ProjectId"),
    ("heldBy",                "PersonSkill",          "Person",       "sv_person_skill",           "PersonSkillKey",        "PersonId"),
    ("refersToSkill",         "PersonSkill",          "Skill",        "sv_person_skill",           "PersonSkillKey",        "SkillId"),
    ("certificationHeldBy",   "PersonCertification",  "Person",       "sv_person_certification",   "PersonCertificationKey", "PersonId"),
    ("refersToCertification", "PersonCertification",  "Certification", "sv_person_certification",  "PersonCertificationKey", "CertificationId"),
    ("availabilityOf",        "Availability",         "Person",       "sv_availability",           "AvailabilityKey",       "PersonId"),
    ("deliveredFor",          "Project",              "Customer",     "sv_project",                "ProjectId",             "CustomerId"),
    ("usedIn",                "ProjectTechnology",    "Project",      "sv_project_technology",     "ProjectTechnologyKey",  "ProjectId"),
    ("usesTechnology",        "ProjectTechnology",    "Technology",   "sv_project_technology",     "ProjectTechnologyKey",  "TechnologyId"),
    ("requiredIn",            "ProjectRequiredSkill", "Project",      "sv_project_required_skill", "ProjectRequiredSkillKey", "ProjectId"),
    ("requiresSkill",         "ProjectRequiredSkill", "Skill",        "sv_project_required_skill", "ProjectRequiredSkillKey", "SkillId"),
    ("producedBy",            "Deliverable",          "Project",      "sv_deliverable",            "DeliverableId",         "ProjectId"),
    ("learnedFrom",           "Knowledge",            "Project",      "sv_knowledge",              "KnowledgeId",           "ProjectId"),
    ("authoredBy",            "Knowledge",            "Person",       "sv_knowledge",              "KnowledgeId",           "AuthorPersonId"),
    ("worksFor",              "Stakeholder",          "Customer",     "sv_stakeholder",            "StakeholderId",         "CustomerId"),
    ("targetsCustomer",       "Opportunity",          "Customer",     "sv_opportunity",            "OpportunityId",         "CustomerId"),
    ("ownedBy",               "Opportunity",          "Person",       "sv_opportunity",            "OpportunityId",         "OwnerPersonId"),
]

#### エンティティ型のメタデータ

記述はすべて英語です。オントロジーは組織横断で共有される語彙層であり、エージェントの
基盤モデルも英語での理解が安定するためです。**データ値そのものが日本語の場合は、
英語の説明文の中で値の意味を明示**しています。

In [ ]:
# ============================================================
# エンティティ型のメタデータ
# ============================================================
# name: (description, [synonyms...], {additional metadata})
#
# 記述はすべて英語で定義する。オントロジーは組織横断・国際的に共有される語彙層であり、
# エージェントの基盤モデルも英語での理解が最も安定するため。
# 説明は「何を表すか」から書き始め、業務上の文脈を含める（1〜3文）。
# シノニムには略称・業界用語・利用者が使いそうな別表現を入れる。
ENTITY_METADATA = {
    "Person": (
        "An engineer or consultant employed by the company. The starting point for staffing "
        "searches and team formation. Skills and availability are held by separate entity "
        "types, so this alone only tells you who exists, not what they can do or when.",
        ["employee", "staff", "member", "engineer", "consultant", "resource",
         "talent", "worker", "practitioner"],
        {"sensitivity": "Contains personal data", "businessOwner": "HR Department",
         "sourceSystem": "Microsoft Entra ID"},
    ),
    "Organization": (
        "An internal department or team. Serves both as the unit an engineer belongs to and "
        "as the domain grouping such as Data & AI or Cloud Infrastructure.",
        ["department", "division", "team", "business unit", "org", "group"],
        {"businessOwner": "HR Department", "sourceSystem": "Microsoft Entra ID"},
    ),
    "Skill": (
        "Master list of technical skills, business skills, and industry knowledge. "
        "Held by engineers and required by projects.",
        ["skill", "capability", "competency", "expertise", "technical skill", "proficiency"],
        {"sourceSystem": "Microsoft Dataverse", "masterData": "true"},
    ),
    "Certification": (
        "Master list of professional and vendor certifications. Used as objective evidence "
        "backing up a claimed skill.",
        ["certification", "credential", "qualification", "accreditation", "vendor certification"],
        {"sourceSystem": "Microsoft Dataverse", "masterData": "true"},
    ),
    "Technology": (
        "Master list of products, services, and technical components. Represents the "
        "technology stack used on a project.",
        ["technology", "product", "platform", "tool", "tech stack", "component"],
        {"sourceSystem": "Microsoft Dataverse", "masterData": "true"},
    ),
    "Customer": (
        "A client company. Carries industry, region, account tier, and business challenges. "
        "Projects and opportunities are delivered against a customer.",
        ["customer", "client", "account", "company", "buyer"],
        {"sensitivity": "Customer confidential", "businessOwner": "Sales Division",
         "sourceSystem": "Salesforce Sales Cloud"},
    ),
    "Stakeholder": (
        "A key person on the customer side. Carries their role, such as decision maker or "
        "IT lead, together with their level of influence.",
        ["stakeholder", "contact", "key person", "decision maker", "customer contact",
         "sponsor"],
        {"sensitivity": "Customer confidential", "businessOwner": "Sales Division",
         "sourceSystem": "Salesforce Sales Cloud"},
    ),
    "Project": (
        "An engagement delivered for a customer. The unit that accumulates experience, with "
        "technologies, deliverables, and lessons learned attached. Because it also carries "
        "revenue and cost, it supports profitability analysis.",
        ["project", "engagement", "deal", "assignment", "initiative", "case", "job"],
        {"sensitivity": "Contains cost and margin data", "businessOwner": "PMO",
         "sourceSystem": "ServiceNow"},
    ),
    "Deliverable": (
        "A document produced during a project, such as a proposal, design specification, or "
        "test plan. The primary target for reuse on new engagements.",
        ["deliverable", "document", "artifact", "specification", "design document",
         "proposal", "output"],
        {"sensitivity": "May contain confidential material", "businessOwner": "PMO",
         "sourceSystem": "SharePoint Online"},
    ),
    "Knowledge": (
        "A lesson learned, best practice, or issue captured from a project. Individually it "
        "is one team's record, but read across projects it becomes organizational learning.",
        ["knowledge", "lesson learned", "best practice", "insight", "know-how",
         "retrospective", "learning"],
        {"businessOwner": "Technology Office", "sourceSystem": "Atlassian Confluence"},
    ),
    "Opportunity": (
        "An active or past sales opportunity. Carries the skills it requires, which makes it "
        "the starting point for team formation planning.",
        ["opportunity", "deal", "pipeline", "prospect", "lead", "sales opportunity"],
        {"sensitivity": "Sales confidential", "businessOwner": "Sales Division",
         "sourceSystem": "Salesforce Sales Cloud"},
    ),
    "Availability": (
        "An engineer's monthly utilization and remaining free capacity. Used in staffing "
        "searches to judge when someone can join. Carries a business time axis (YearMonth), "
        "with one row per person per month.",
        ["availability", "capacity", "free capacity", "utilization", "bench",
         "allocation", "bandwidth", "open capacity"],
        {"unit": "percent", "grain": "monthly", "businessOwner": "PMO",
         "sourceSystem": "Dynamics 365 Project Operations",
         "freshness": "Check the DataAsOfDate property for the reference date"},
    ),
    "Assignment": (
        "An engineer's record of having worked on a project. Carries the role played, "
        "allocation percentage, and contribution outcome. This is the evidence for who has "
        "experience with what.",
        ["assignment", "staffing", "participation", "project experience", "allocation",
         "engagement record"],
        {"unit": "AllocationPercent is a percentage", "businessOwner": "PMO",
         "sourceSystem": "Dynamics 365 Project Operations"},
    ),
    "PersonSkill": (
        "A skill held by an engineer, together with proficiency level. The EvidenceType "
        "property distinguishes self-declared skills from skills backed by project "
        "experience, which matters when judging how much to trust the data.",
        ["skill held", "person skill", "capability held", "proficiency", "skill record"],
        {"businessOwner": "HR Department", "sourceSystem": "SAP SuccessFactors",
         "dataQuality": "Rows where EvidenceType is self-declared have no supporting evidence"},
    ),
    "PersonCertification": (
        "A certification obtained by an engineer, with the acquisition date and expiry date.",
        ["certification held", "credential held", "person certification", "qualification record"],
        {"businessOwner": "HR Department", "sourceSystem": "SAP SuccessFactors"},
    ),
    "ProjectTechnology": (
        "A technology used on a project, distinguishing primary technologies from "
        "supporting ones.",
        ["technology used", "tech used", "project technology", "adopted technology"],
        {"businessOwner": "PMO", "sourceSystem": "ServiceNow"},
    ),
    "ProjectRequiredSkill": (
        "A skill required by a project. Used as the required-skill checklist when planning "
        "a delivery team.",
        ["required skill", "skill requirement", "needed skill", "demanded skill"],
        {"businessOwner": "PMO", "sourceSystem": "ServiceNow"},
    ),
}

#### プロパティのメタデータ

In [ ]:
# ============================================================
# プロパティのメタデータ（説明のみ。シノニムは非対応）
# ============================================================
# エンティティ型名 -> {プロパティ名: (description, {additional metadata})}
# 全プロパティに付ける必要はない。意味が自明でないもの、単位があるもの、
# 機密区分があるものに絞る。
PROPERTY_METADATA = {
    "Person": {
        "YearsOfService": ("Years since joining the company. Note that this is distinct from "
                           "years of experience with a specific skill.",
                           {"unit": "years"}),
        "Email": ("Work email address. Personal data, so handle with care when exporting "
                  "outside the organization.",
                  {"sensitivity": "Personal data"}),
    },
    "Project": {
        "SolutionDomain": ("The technical domain of the project. One of AI, データ基盤 "
                           "(data platform), クラウド (cloud), BI, ガバナンス (governance), "
                           "コンサル (consulting), or 基幹 (core systems). Note that only AI "
                           "is stored in the Latin alphabet. To find generative AI "
                           "engagements, look for rows where this value is AI.", {}),
        "Status": ("Progress state of the project. One of Completed, In Progress, or Planned "
                   "(stored in Japanese as 完了 / 進行中 / 計画中).", {}),
        "RevenueJPY": ("Revenue booked for the project.",
                       {"unit": "JPY", "sensitivity": "Financial data"}),
        "CostJPY": ("Delivery cost of the project.",
                    {"unit": "JPY", "sensitivity": "Financial data, access controlled"}),
        "MarginRate": ("Gross margin rate as a decimal. A value of 0.25 means 25 percent.",
                       {"unit": "ratio", "sensitivity": "Financial data, access controlled"}),
        "ContractType": ("Contract type (stored in Japanese): 準委任 is a quasi-mandate "
                         "contract billed on effort, 請負 is a fixed-price contract where the "
                         "delivery risk sits with us, and ライセンス+SI bundles licence resale "
                         "with system integration. Margin differs meaningfully by type.", {}),
        "DevelopmentMethod": ("Delivery approach (stored in Japanese): アジャイル (agile), "
                              "ウォーターフォール (waterfall), or ハイブリッド (hybrid). Useful "
                              "when analysing why margin differs within the same domain.", {}),
        "PlatformType": ("Target platform of the delivered system (stored in Japanese).", {}),
    },
    "Customer": {
        "Industry": ("Industry classification of the customer, stored in Japanese. The values "
                     "in use are 製造 (manufacturing), 流通 (retail and distribution), 金融 "
                     "(financial services), 公共 (public sector), エネルギー (energy), 物流 "
                     "(logistics), and ヘルスケア (healthcare). Use this value to find "
                     "customers in a given industry.", {}),
        "AccountTier": ("Account classification indicating sales priority, stored in Japanese: "
                        "戦略顧客 (strategic), 重点顧客 (key), or 一般顧客 (general).", {}),
        "AnnualITBudgetOkuJPY": ("Annual IT budget of the customer.",
                                 {"unit": "hundred million JPY",
                                  "sensitivity": "Customer confidential"}),
        "PrimaryBusinessChallenge": ("The main business challenge the customer faces. The "
                                     "starting point for shaping a proposal.", {}),
    },
    "Availability": {
        "YearMonth": ("Target month in YYYY-MM format. The business time axis indicating "
                      "which month this capacity row describes.", {}),
        "AllocatedPercent": ("Share of capacity already assigned for that month.",
                             {"unit": "percent"}),
        "AvailablePercent": ("Share of capacity still free for that month, equal to 100 minus "
                             "AllocatedPercent. Staffing searches typically filter on "
                             "conditions such as at least 30 percent free.",
                             {"unit": "percent"}),
        "AvailabilityStatus": ("Capacity status on a three-step scale (stored in Japanese): "
                               "空きあり means 30 percent or more is free, 調整可 means 11 to 29 "
                               "percent is free and the person could be freed up by adjusting "
                               "assignments, 逼迫 means 10 percent or less is free. When asked "
                               "for people who are available, 調整可 rows are worth surfacing "
                               "as a second tier.", {}),
        "DataAsOfDate": ("The date this capacity information reflects. Capacity changes daily, "
                         "so answers should state the reference date.",
                         {"format": "YYYY-MM-DD"}),
    },
    "PersonSkill": {
        "ProficiencyLevel": ("Proficiency on a four-step scale from beginner to expert "
                             "(stored in Japanese as 初級 / 中級 / 上級 / エキスパート).", {}),
        "YearsOfExperience": ("Years of hands-on experience with this specific skill. Distinct "
                              "from the person's years of service.", {"unit": "years"}),
        "EvidenceProjectId": ("The project that substantiates this skill. Empty when the skill "
                              "is only self-declared.", {}),
        "EvidenceType": ("How the skill is substantiated. A value of 案件実績 means it is backed "
                         "by project experience; 自己申告 means it is self-declared only. "
                         "Filter on project-backed rows when higher confidence is needed.", {}),
    },
    "Assignment": {
        "ProjectRole": ("Role played on the project, stored in Japanese except for PM. The "
                        "values are PM, アーキテクト (architect), テックリード (tech lead), "
                        "データエンジニア (data engineer), AIエンジニア (AI engineer), アプリ開発 "
                        "(application development), 業務コンサルタント (business consultant), "
                        "インフラエンジニア (infrastructure engineer), and 品質管理 (quality "
                        "assurance). The first three are the lead roles worth asking first.", {}),
        "AllocationPercent": ("Allocation to the project at the time of participation.",
                              {"unit": "percent"}),
        "ContributionResult": ("Evaluation of the contribution, stored in Japanese: 高評価 "
                               "(high performance), 目標達成 (met expectations), or 課題あり "
                               "(issues raised).", {}),
    },
    "Knowledge": {
        "KnowledgeType": ("Category of the record, stored in the Latin alphabet: "
                          "LessonsLearned, BestPractice, or Issue. LessonsLearned and Issue "
                          "both describe things that went wrong; BestPractice describes what "
                          "worked.", {}),
        "ReuseValue": ("How valuable this record is for reuse, stored in Japanese: 高 (high), "
                       "中 (medium), or 低 (low).", {}),
        "Tags": ("Search tags for the record, stored as a comma-separated string. The same "
                 "underlying issue always carries the same tag string even when the wording "
                 "of the summary differs between authors, so grouping by this property is the "
                 "reliable way to count how often an issue recurs.",
                 {"format": "comma-separated"}),
    },
    "Deliverable": {
        "Confidentiality": ("Confidentiality classification, stored in Japanese: 社外秘 means "
                            "it must not leave the company, 社内限 means internal use only. "
                            "Always check this before sharing a deliverable outside the "
                            "organization.",
                            {"sensitivity": "Drives access control decisions"}),
        "ReusabilityRating": ("How readily this deliverable can be reused, stored in Japanese: "
                              "高 (high), 中 (medium), or 低 (low). Filter on 高 when looking "
                              "for material worth reusing in a new proposal.", {}),
        "DocumentUrl": ("Storage location of the deliverable.", {}),
    },
    "Opportunity": {
        "SalesStage": ("Sales stage of the opportunity, stored in Japanese and ordered as "
                       "情報収集 (research), 提案準備 (preparing a proposal), 提案中 (proposal "
                       "submitted), 最終交渉 (final negotiation), then 受注 (won) or 失注 "
                       "(lost).", {}),
        "ExpectedRevenueJPY": ("Expected revenue if the opportunity is won.",
                               {"unit": "JPY", "sensitivity": "Sales confidential"}),
        "RequiredSkillIds": ("Required skill identifiers stored as a semicolon-separated "
                             "string. Because it is not normalized, it cannot be joined "
                             "directly to the Skill entity type.",
                             {"format": "semicolon-separated",
                              "dataQuality": "Not normalized"}),
    },
}

#### 関係型のメタデータ

In [ ]:
# ============================================================
# 関係型のメタデータ（説明のみ。シノニムは非対応）
# ============================================================
RELATIONSHIP_METADATA = {
    "belongsTo": ("Links an engineer to the organization they belong to. Reflects the current "
                  "assignment after any internal transfer.", {}),
    "performedBy": ("Links a project participation record to the engineer who performed it.", {}),
    "assignedTo": ("Links a project participation record to the project it was performed on. "
                   "Use this to trace which engagements a person has experience with.", {}),
    "heldBy": ("Links a skill record to the engineer who holds it.", {}),
    "refersToSkill": ("Links a skill record to the skill it refers to.", {}),
    "certificationHeldBy": ("Links a certification record to the engineer who holds it.", {}),
    "refersToCertification": ("Links a certification record to the certification it refers to.", {}),
    "availabilityOf": ("Links a monthly capacity record to the engineer it describes. The "
                       "starting point for finding who is free in a given month.", {}),
    "deliveredFor": ("Links a project to the customer it was delivered for. Use this to trace "
                     "engagements within a given industry.", {}),
    "usedIn": ("Links a technology usage record to the project it belongs to.", {}),
    "usesTechnology": ("Links a technology usage record to the technology it refers to.", {}),
    "requiredIn": ("Links a required-skill record to the project it belongs to.", {}),
    "requiresSkill": ("Links a required-skill record to the skill it refers to.", {}),
    "producedBy": ("Links a deliverable to the project that produced it.", {}),
    "learnedFrom": ("Links a knowledge record to the project it was learned from.", {}),
    "authoredBy": ("Links a knowledge record to the person who captured it. Useful for finding "
                   "an internal expert to consult.", {}),
    "worksFor": ("Links a customer-side stakeholder to the customer company they work for.", {}),
    "targetsCustomer": ("Links an opportunity to the customer it targets.", {}),
    "ownedBy": ("Links an opportunity to the salesperson who owns it.", {}),
}

#### ヘルパー

In [ ]:
# ============================================================
# ヘルパー
# ============================================================
def stable_id(*parts):
    """名前から決定論的に正の64bit整数IDを生成する（再実行しても同じ値）。"""
    h = uuid.uuid5(uuid.NAMESPACE_OID, "|".join(parts)).int
    return str((h % (2**62 - 1)) + 1)


def stable_guid(*parts):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, "|".join(parts)))


def b64(obj):
    return base64.b64encode(json.dumps(obj, ensure_ascii=False).encode()).decode()


def unb64(payload):
    return json.loads(base64.b64decode(payload).decode())


def api(method, path, **kwargs):
    r = requests.request(method, f"{FABRIC_API}{path}", headers=HEADERS, **kwargs)

    # 長時間実行（LRO）の場合は完了まで待つ。
    # 公式仕様では 202 のとき Location / x-ms-operation-id / Retry-After が返る。
    # ただし Location は analytics 側のホストに転送され認証に失敗するため使わず、
    # /v1/operations/{id} をポーリングし、結果は /result から取得する。
    # 待ち時間は Retry-After に従う（既定 30 秒だとデモには長いので上限を設ける）。
    if r.status_code == 202:
        op = r.headers.get("x-ms-operation-id")
        wait = min(int(r.headers.get("Retry-After") or 3), 10)
        for _ in range(60):
            time.sleep(wait)
            st = requests.get(f"{FABRIC_API}/operations/{op}", headers=HEADERS).json()
            if st.get("status") == "Failed":
                raise RuntimeError(f"操作が失敗しました: {st}")
            if st.get("status") == "Succeeded":
                res = requests.get(f"{FABRIC_API}/operations/{op}/result", headers=HEADERS)
                return res.json() if res.ok and res.text else st
        raise TimeoutError("操作がタイムアウトしました")

    if not r.ok:
        raise RuntimeError(f"{method} {path} -> {r.status_code}: {r.text[:600]}")
    return r.json() if r.text else {}


# 公式リファレンスのアイテム種別ごとのコレクション名。
#   Ontology   -> /workspaces/{id}/ontologies      （List Ontologies）
#   Lakehouse  -> /workspaces/{id}/lakehouses      （List Lakehouses）
#   GraphModel -> /workspaces/{id}/graphModels     （List Graph Models）
ITEM_COLLECTIONS = {
    "Ontology": "ontologies",
    "Lakehouse": "lakehouses",
    "GraphModel": "graphModels",
}


def list_items(item_type):
    """指定種別のアイテムを全件返す。

    公式の一覧 API はページングされ、続きがある場合だけ continuationUri
    （次ページの完全な URL）が返る。自前でトークンを URL エンコードすると
    二重エンコードで壊れるため、返ってきた URI をそのまま辿る。
    """
    path = f"/workspaces/{WORKSPACE_ID}/{ITEM_COLLECTIONS[item_type]}"
    items = []
    while path:
        res = api("GET", path)
        items += res.get("value", [])
        nxt = res.get("continuationUri")
        path = nxt[len(FABRIC_API):] if nxt and nxt.startswith(FABRIC_API) else None
    return items


def find_item(name, item_type):
    for it in list_items(item_type):
        if it["displayName"] == name:
            return it
    raise LookupError(
        f"{item_type} '{name}' が見つかりません。"
        f" 名前が正しいか、同じワークスペースにあるか確認してください。"
    )

#### 1. 対象アイテムを解決する

In [ ]:
# ============================================================
# 1. 対象アイテムを解決する
# ============================================================
ontology = find_item(ONTOLOGY_NAME, "Ontology")
silver = find_item(SILVER_LAKEHOUSE_NAME, "Lakehouse")
ONTOLOGY_ID, SILVER_ID = ontology["id"], silver["id"]

print(f"ワークスペース : {WORKSPACE_ID}")
print(f"オントロジー   : {ONTOLOGY_NAME} ({ONTOLOGY_ID})")
print(f"シルバー層     : {SILVER_LAKEHOUSE_NAME} ({SILVER_ID})")

#### 2. 既存の定義を取得する

手動で作った分を壊さないよう、現在の定義をすべて読み込みます。

In [ ]:
# ============================================================
# 2. 既存の定義を取得する（手動で作った3つを壊さないため）
# ============================================================
current = api("POST", f"/workspaces/{WORKSPACE_ID}/ontologies/{ONTOLOGY_ID}/getDefinition")
parts = current["definition"]["parts"]

existing_entities = {}   # name -> (id, {property name -> property id})
entities_without_key = []  # キー（entityIdParts）が未設定のエンティティ型
for p in parts:
    if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"):
        d = unb64(p["payload"])
        props = {pr["name"]: pr["id"] for pr in d.get("properties", [])}
        existing_entities[d["name"]] = (d["id"], props)
        if not d.get("entityIdParts"):
            entities_without_key.append(d["name"])

existing_rel_names = {
    unb64(p["payload"])["name"]
    for p in parts
    if p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json")
}

print(f"\n既存のエンティティ型 ({len(existing_entities)}): {', '.join(sorted(existing_entities)) or 'なし'}")
print(f"既存の関係型 ({len(existing_rel_names)}): {', '.join(sorted(existing_rel_names)) or 'なし'}")

if entities_without_key:
    print(f"\n[注意] キーが未設定のエンティティ型: {', '.join(sorted(entities_without_key))}")
    print("       UI でエンティティ型キー（エンティティ ID）を選ばずに保存すると、この状態になります。")
    print("       この状態のエンティティ型を起点にした関係型は作成できないため、")
    print("       次のセルで ENTITY_TYPES の定義に従ってキーを補います。")

if not existing_entities:
    print("\n[注意] エンティティ型が一つもありません。")
    print("       Step 2 の手順どおり、UI から5個のエンティティ型と3個の関係型を")
    print("       作成してから実行してください。")
    print("       このまま続行すると、17個すべてをコードで作成します。")

#### フェーズ1 — 構造をつくる

まだ無いエンティティ型と関係型を、データバインドつきで定義に追加します。

In [ ]:
# ============================================================
# フェーズ1: 構造をつくる（エンティティ型・関係型・データバインド）
# ============================================================
def property_id(entity_name, prop_name):
    """既存のプロパティ ID があればそれを使い、無ければ生成する。"""
    if entity_name in existing_entities and prop_name in existing_entities[entity_name][1]:
        return existing_entities[entity_name][1][prop_name]
    return stable_id("prop", entity_name, prop_name)


def entity_id(entity_name):
    if entity_name in existing_entities:
        return existing_entities[entity_name][0]
    return stable_id("entity", entity_name)


new_parts = []
created_entities = []

# ------------------------------------------------------------
# 補修: キー（entityIdParts）が未設定の既存エンティティ型を直す
# ------------------------------------------------------------
# UI でエンティティ型を作るとき、エンティティ型キーを選ばずに保存できてしまいます。
# その状態のエンティティ型を関係型の起点にすると、定義の更新が次のエラーで失敗します。
#
#   Contextualization sourceKeyRefBindings count (1) must match
#   the number of EntityIdParts (0) in the source EntityType '...'
#
# 関係の起点は「起点エンティティのキー」で突き合わせるため、キーが0個だと
# 突き合わせ列を1個渡しても数が合わない、という意味です。
# ここで ENTITY_TYPES の定義に従ってキーを補い、UI に戻らずに解消します。
repaired_entities = []

for idx, p in enumerate(parts):
    if not (p["path"].startswith("EntityTypes/")
            and p["path"].endswith("/definition.json")):
        continue

    d = unb64(p["payload"])
    name = d.get("name")
    if name not in ENTITY_TYPES or d.get("entityIdParts"):
        continue

    key_prop = ENTITY_TYPES[name][1]
    props = {pr["name"]: pr["id"] for pr in d.get("properties", [])}

    if key_prop not in props:
        raise RuntimeError(
            f"エンティティ型 '{name}' にキー '{key_prop}' がありません。\n"
            f"  現在のプロパティ: {', '.join(sorted(props)) or 'なし'}\n"
            f"  UI で '{name}' を開き、'{key_prop}' をバインドしてから"
            f"エンティティ型キーに選び、保存してください。"
        )

    d["entityIdParts"] = [props[key_prop]]
    if not d.get("displayNamePropertyId"):
        display_prop = DISPLAY_NAME_PROPERTY.get(name, key_prop)
        d["displayNamePropertyId"] = props.get(display_prop, props[key_prop])

    parts[idx] = {"path": p["path"], "payload": b64(d),
                  "payloadType": "InlineBase64"}
    repaired_entities.append(f"{name}（キー: {key_prop}）")

if repaired_entities:
    print(f"\nキーを補ったエンティティ型 ({len(repaired_entities)}): "
          f"{', '.join(repaired_entities)}")

for name, (table, key_prop, props) in ENTITY_TYPES.items():
    if name in existing_entities:
        continue  # 手動で作成済み。触らない

    eid = entity_id(name)
    prop_defs = [
        {
            "id": property_id(name, pname),
            "name": pname,
            "redefines": None,
            "baseTypeNamespaceType": None,
            "valueType": ptype,
        }
        for pname, ptype in props
    ]
    display_prop = DISPLAY_NAME_PROPERTY.get(name, key_prop)

    new_parts.append({
        "path": f"EntityTypes/{eid}/definition.json",
        "payload": b64({
            "id": eid,
            "namespace": "usertypes",
            "baseEntityTypeId": None,
            "name": name,
            "entityIdParts": [property_id(name, key_prop)],
            "displayNamePropertyId": property_id(name, display_prop),
            "namespaceType": "Custom",
            "visibility": "Visible",
            "properties": prop_defs,
            "timeseriesProperties": [],
        }),
        "payloadType": "InlineBase64",
    })

    # データバインド（シルバー層のテーブルに接続）
    binding_id = stable_guid("binding", name, table)
    new_parts.append({
        "path": f"EntityTypes/{eid}/DataBindings/{binding_id}.json",
        "payload": b64({
            "id": binding_id,
            "dataBindingConfiguration": {
                "dataBindingType": "NonTimeSeries",
                "propertyBindings": [
                    {"sourceColumnName": pname, "targetPropertyId": property_id(name, pname)}
                    for pname, _ in props
                ],
                "sourceTableProperties": {
                    "sourceType": "LakehouseTable",
                    "workspaceId": WORKSPACE_ID,
                    "itemId": SILVER_ID,
                    "sourceTableName": table,
                    **({"sourceSchema": SILVER_SCHEMA} if SILVER_SCHEMA else {}),
                },
            },
        }),
        "payloadType": "InlineBase64",
    })
    created_entities.append(name)

# --- 関係型 ---
created_rels = []
for rel_name, src, tgt, table, src_col, tgt_col in RELATIONSHIP_TYPES:
    if rel_name in existing_rel_names:
        continue

    src_key = ENTITY_TYPES[src][1]
    # 起点のキープロパティが存在しないと、突き合わせの数が合わずに更新が 400 になる
    if src in existing_entities and src_key not in existing_entities[src][1]:
        raise RuntimeError(
            f"関係型 '{rel_name}' の起点 '{src}' に、キー '{src_key}' がありません。\n"
            f"  UI で '{src}' に '{src_key}' をバインドし、"
            f"エンティティ型キーに選んでから再実行してください。"
        )

    tgt_key = ENTITY_TYPES[tgt][1]
    rid = stable_id("rel", rel_name, src, tgt)

    new_parts.append({
        "path": f"RelationshipTypes/{rid}/definition.json",
        "payload": b64({
            "namespace": "usertypes",
            "id": rid,
            "name": rel_name,
            "namespaceType": "Custom",
            "source": {"entityTypeId": entity_id(src)},
            "target": {"entityTypeId": entity_id(tgt)},
        }),
        "payloadType": "InlineBase64",
    })

    ctx_id = stable_guid("ctx", rel_name, src, tgt)
    new_parts.append({
        "path": f"RelationshipTypes/{rid}/Contextualizations/{ctx_id}.json",
        "payload": b64({
            "id": ctx_id,
            "dataBindingTable": {
                "workspaceId": WORKSPACE_ID,
                "itemId": SILVER_ID,
                "sourceTableName": table,
                "sourceType": "LakehouseTable",
                **({"sourceSchema": SILVER_SCHEMA} if SILVER_SCHEMA else {}),
            },
            # 起点側: マッピングテーブルの列 → 起点エンティティのキープロパティ
            "sourceKeyRefBindings": [
                {"sourceColumnName": src_col, "targetPropertyId": property_id(src, src_key)}
            ],
            # 終点側: マッピングテーブルの列 → 終点エンティティのキープロパティ
            "targetKeyRefBindings": [
                {"sourceColumnName": tgt_col, "targetPropertyId": property_id(tgt, tgt_key)}
            ],
        }),
        "payloadType": "InlineBase64",
    })
    created_rels.append(rel_name)

print(f"\n追加するエンティティ型 ({len(created_entities)}): {', '.join(created_entities) or 'なし'}")
print(f"追加する関係型 ({len(created_rels)}): {', '.join(created_rels) or 'なし'}")


# ============================================================
# 診断: 定義の中身をそのまま出力する（DUMP_DEFINITION = True で有効）
# ============================================================
# セマンティックエンリッチメントの格納場所は公式ドキュメントに記載がありません。
# メタデータが検出されない場合は、DUMP_DEFINITION = True にして実行すると、
# 定義ツリー全体のパスと中身が出力されます。
#
# 確実な特定手順:
#   1. UI でメタデータを設定する *前* に、DUMP_DEFINITION = True で実行して出力を控える
#   2. UI で Person に説明・シノニム・追加メタデータを設定する
#   3. もう一度 DUMP_DEFINITION = True で実行し、増えたパート／増えたフィールドを見る
#   これが公式に記載がない現状で、格納場所を確定できる唯一の方法です。

DUMP_DEFINITION = False

if DUMP_DEFINITION:
    print("=" * 70)
    print(f"定義ツリー全体（{len(parts)} パート）")
    print("=" * 70)
    for p in sorted(parts, key=lambda x: x["path"]):
        print(f"\n[{p['path']}]")
        try:
            body = json.dumps(unb64(p["payload"]), ensure_ascii=False, indent=2)
        except Exception:
            body = "(JSON として解釈できません)"
        print(body[:1500])
    print("\n" + "=" * 70)

#### フェーズ2 — 意味を与える

各定義の `semanticEnrichment` にメタデータを書き込みます。`OVERWRITE_EXISTING_METADATA`
が `False` のとき、UI で設定済みの内容は上書きしません。

In [ ]:
# ============================================================
# フェーズ2: 意味を与える（セマンティックエンリッチメント）
# ============================================================
# ■ 格納場所とフィールド名（確定仕様）
# セマンティックエンリッチメントは、エンティティ型・プロパティ・関係型それぞれの
# definition.json の中に、"semanticEnrichment" というオブジェクトとして格納されます。
# 公式のスキーマ（entityType/1.0.0/schema.json）には記載がありませんが、
# 実際の Fabric はこの形で受け付けます。
#
#   EntityTypes/{id}/definition.json
#     {
#       "id": "...", "name": "Person", "properties": [
#         { "id": "...", "name": "FullName", "valueType": "String",
#           "semanticEnrichment": {                      ← プロパティ単位
#             "description": "...",
#             "customAttributes": { "unit": "years" }    ← key-value は辞書
#           } }
#       ],
#       "semanticEnrichment": {                          ← エンティティ型単位
#         "description": "...",
#         "synonyms": ["employee", "staff"],             ← エンティティ型のみ
#         "customAttributes": { "sensitivity": "..." }
#       }
#     }
#
#   RelationshipTypes/{id}/definition.json
#     { "id": "...", "name": "belongsTo", "source": {...}, "target": {...},
#       "semanticEnrichment": { "description": "...", "customAttributes": {...} } }
#
# 重要な点:
#   - key-value は **辞書**（[{"key":..,"value":..}] のリスト形式ではない）
#   - シノニムはエンティティ型のみ。プロパティと関係型には付けない
#   - 追加メタデータのフィールド名は "additionalMetadata" ではなく "customAttributes"
#   - 構造（id / name / valueType / source / target）は一切変更しない

SEMANTIC_KEY = "semanticEnrichment"


def build_enrichment(description, synonyms=None, custom=None):
    """semanticEnrichment オブジェクトを組み立てる。"""
    obj = {"description": description}
    if synonyms:
        obj["synonyms"] = list(synonyms)
    if custom:
        # 値はすべて文字列に統一する（数値や真偽値はそのままだと弾かれることがある）
        obj["customAttributes"] = {str(k): str(v) for k, v in custom.items()}
    return obj


def set_enrichment(target, desired, label, stat, kind):
    """既存と異なる場合のみ semanticEnrichment を差し替える。"""
    current = target.get(SEMANTIC_KEY)
    if current == desired:
        stat["unchanged"] += 1
        return False
    if current and not OVERWRITE_EXISTING_METADATA:
        # 手動設定済み。上書きしない
        stat["skipped"] += 1
        return False
    target[SEMANTIC_KEY] = desired
    stat[kind] += 1
    return True


all_parts = parts + new_parts
final_parts = []
meta_stat = {"entity": 0, "property": 0, "relationship": 0,
             "skipped": 0, "unchanged": 0}

if APPLY_METADATA:
    for p in all_parts:
        path, payload = p["path"], p["payload"]

        # ---- エンティティ型とそのプロパティ ----
        if path.startswith("EntityTypes/") and path.endswith("/definition.json"):
            d = unb64(payload)
            name = d.get("name")
            touched = False

            if name in ENTITY_METADATA:
                desc, syns, custom = ENTITY_METADATA[name]
                if set_enrichment(d, build_enrichment(desc, syns, custom),
                                  name, meta_stat, "entity"):
                    touched = True

            # プロパティ単位（静的プロパティと時系列プロパティの両方）
            for collection in ("properties", "timeseriesProperties"):
                for prop in d.get(collection, []):
                    pm = PROPERTY_METADATA.get(name, {}).get(prop.get("name"))
                    if not pm:
                        continue
                    if set_enrichment(prop, build_enrichment(pm[0], None, pm[1]),
                                      f"{name}.{prop.get('name')}",
                                      meta_stat, "property"):
                        touched = True

            if touched:
                final_parts.append({"path": path, "payload": b64(d),
                                    "payloadType": "InlineBase64"})
                continue

        # ---- 関係型 ----
        elif path.startswith("RelationshipTypes/") and path.endswith("/definition.json"):
            d = unb64(payload)
            rm = RELATIONSHIP_METADATA.get(d.get("name"))
            if rm and set_enrichment(d, build_enrichment(rm[0], None, rm[1]),
                                     d.get("name"), meta_stat, "relationship"):
                final_parts.append({"path": path, "payload": b64(d),
                                    "payloadType": "InlineBase64"})
                continue

        final_parts.append(p)

    print("\nメタデータ適用:")
    print(f"  エンティティ型 : {meta_stat['entity']} 件")
    print(f"  プロパティ     : {meta_stat['property']} 件")
    print(f"  関係型         : {meta_stat['relationship']} 件")
    if meta_stat["unchanged"]:
        print(f"  変更なし       : {meta_stat['unchanged']} 件（すでに同じ内容）")
    if meta_stat["skipped"]:
        print(f"  スキップ       : {meta_stat['skipped']} 件"
              f"（手動設定済み。上書きするには OVERWRITE_EXISTING_METADATA = True）")
else:
    final_parts = all_parts
    print("\nAPPLY_METADATA = False のため、メタデータの適用をスキップします。")

#### フェーズ3 — 定義を更新する

**`updateDefinition` は送ったパートで定義を置き換えます。** 送らなかったパートは
消えるため、取得した全パートをそのまま含めて送ります。

In [ ]:
# ============================================================
# フェーズ3: 定義を更新する
# ============================================================
# updateDefinition は「含めたパートで現在の定義を置き換える」API です。
# 送らなかったパートは消えるため、必ず取得した全パート（parts）を含めて送ります。
# このノートブックは final_parts に既存分をすべて含めているため、
# UI で手動設定したメタデータが消えることはありません。
n_meta = meta_stat["entity"] + meta_stat["property"] + meta_stat["relationship"]

if not new_parts and n_meta == 0 and not repaired_entities:
    print("\n追加・変更するものはありません。すでに構築済みです。")
else:
    # 既存分をそのまま含めないと消えるため、全パートを送る
    print(f"\n定義を更新しています（parts: {len(parts)} -> {len(final_parts)}）...")
    api("POST", f"/workspaces/{WORKSPACE_ID}/ontologies/{ONTOLOGY_ID}/updateDefinition",
        json={"definition": {"parts": final_parts}})
    print("更新しました。")

#### 結果を確認する

In [ ]:
# ============================================================
# 結果を確認する
# ============================================================
after = api("POST", f"/workspaces/{WORKSPACE_ID}/ontologies/{ONTOLOGY_ID}/getDefinition")
ap = after["definition"]["parts"]
ents = sorted(
    unb64(p["payload"])["name"] for p in ap
    if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json")
)
rels = sorted(
    unb64(p["payload"])["name"] for p in ap
    if p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json")
)

ent_meta = []
n_prop_desc = 0
rel_meta = []
for p in ap:
    if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"):
        d = unb64(p["payload"])
        se = d.get("semanticEnrichment") or {}
        ent_meta.append((d["name"], bool(se.get("description")),
                         len(se.get("synonyms") or [])))
        for collection in ("properties", "timeseriesProperties"):
            n_prop_desc += sum(
                1 for pr in d.get(collection, [])
                if (pr.get("semanticEnrichment") or {}).get("description")
            )
    elif p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json"):
        d = unb64(p["payload"])
        se = d.get("semanticEnrichment") or {}
        rel_meta.append((d["name"], bool(se.get("description"))))

print("\n" + "=" * 70)
print(f"エンティティ型 {len(ents)} 個（想定 17）")
print(f"  {'名前':24s} {'説明':>4s} {'シノニム':>8s}")
for n, has_d, n_syn in sorted(ent_meta):
    print(f"  {n:24s} {'OK' if has_d else '--':>4s} {n_syn:>6d} 件")

print(f"\n関係型 {len(rels)} 個（想定 19）")
n_rel_desc = sum(1 for _, h in rel_meta if h)
print(f"  説明あり: {n_rel_desc} / {len(rel_meta)} 件")
missing_rel = [n for n, h in rel_meta if not h]
if missing_rel:
    print(f"  未設定: {', '.join(missing_rel)}")

print(f"\nプロパティの説明: {n_prop_desc} 件（想定 35）")
print("=" * 70)

if len(ents) == 17 and len(rels) == 19:
    print("\n構築完了。Fabric でオントロジーを開き、構成キャンバスで確認してください。")
    print("エンティティ型の詳細を開くと、メタデータ欄に説明とシノニムが入っています。")
    if True:
        print("Step 6 のシナリオで、シノニム（例：Availability の \"bandwidth\"）が")
        print("通るようになっているかを試すと効果が分かります。")
    print("この後 Step 3（MCP エンドポイント）に進めます。")
else:
    print("\n[注意] 想定と数が違います。手動作成分の名前を確認してください。")
    print("       エンティティ型: Person / Project / Customer / Organization / Assignment")
    print("       関係型: belongsTo / performedBy / deliveredFor")
    print("       この綴りで作成している必要があります（大文字小文字も一致させること）。")

#### フェーズ4 — グラフを構築する

**ここが MCP でデータ検索を動かすために必須です。**

オントロジーを作成すると、実データの格納と検索を担当する**グラフ**が子アイテムとして
自動生成されます。公式ドキュメントには、スキーマを変更するたびにバインド済みの全データが
自動で再取り込みされる、と書かれています。ただしこれは UI で操作した場合の動作です。

**本ノートブックのように REST API でオントロジーを構築すると、この自動反映が起こらず、
グラフが空のまま残ります。** 空のグラフはリフレッシュも拒否され
（`GraphNotRefreshable`）、MCP に接続してもスキーマは見えるのにデータ検索がすべて
失敗する、という状態になります。デモで最も遭遇しやすい失敗がこれです。

このフェーズは、オントロジーのデータバインドをそのままグラフの定義に翻訳し、
取り込みまで実行することでこれを解消します。

In [ ]:
# ============================================================
# フェーズ4: グラフを構築する（Graph in Microsoft Fabric）
#
# オントロジーを作成すると、子アイテムとして GraphModel が自動生成され、
# 実データの格納と検索を担当します。MCP のデータ検索もここを参照します。
#
# 公式には「スキーマを変更するたびに、バインド済みの全データが自動で
# 再取り込みされる」とされていますが、これは UI 操作を前提とした動作です。
# REST API の updateDefinition でオントロジーを構築した場合、この自動反映が
# 起こらず、GraphModel が空のまま残ります。空のグラフはリフレッシュも
# 拒否され（GraphNotRefreshable）、MCP からのデータ検索が全て失敗します。
#
# そこで、オントロジーの定義（エンティティ型・関係型とそのバインド）から
# グラフの定義を機械的に生成し、GraphModel に直接書き込みます。
#
# GraphModel の定義スキーマは公開されていないため、以下は実測値です。
#   プロパティ型   : STRING / INT / FLOAT
#   データソース型 : DeltaTable（テーブル1つにつき1つ）
#   パス           : abfss://{ws}@onelake.dfs.fabric.microsoft.com
#                      /{item}/Tables/{schema}/{table}
#   ノード型       : name / alias / labels / primaryKeyProperties / properties
#   ノードテーブル : id / nodeType / nodeTypeAlias / dataSourceId /
#                    dataSourceName / schemaName / tableName /
#                    keyColumns / propertyMappings[{sourceColumn, propertyName}]
#   エッジ型       : alias / labels / properties /
#                    sourceNodeType{alias} / destinationNodeType{alias}
#   エッジテーブル : id / edgeTypeAlias / dataSourceName /
#                    sourceNodeKeyColumns / destinationNodeKeyColumns
#
# 「target」ではなく「destination」を使う点に注意してください。
# 検証は寛容で、知らないフィールドは黙って捨てられます。結合列の名前を
# 間違えると、定義の更新は成功するのに取り込みだけが失敗します。
# ============================================================
BUILD_GRAPH = True                # グラフの構築まで行うか
WAIT_FOR_REFRESH = True           # 取り込みの完了を待つか
REFRESH_TIMEOUT_SEC = 1800        # 取り込みの待ち時間の上限

GRAPH_SCHEMA_BASE = ("https://developer.microsoft.com/json-schemas/fabric/item"
                     "/graphIndex/definition")

# オントロジーの valueType から、グラフのプロパティ型への対応
GRAPH_PROP_TYPES = {
    "String": "STRING",
    "BigInt": "INT",
    "Int": "INT",
    "Double": "FLOAT",
    "Float": "FLOAT",
    "Boolean": "BOOL",
    "DateTime": "DATETIME",
    "Date": "DATE",
}


def onelake_path(ws_id, item_id, schema, table):
    base = f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{item_id}/Tables"
    return f"{base}/{schema}/{table}" if schema else f"{base}/{table}"


_columns_cache = {}


def actual_columns(path):
    """実テーブルの列名を読む。オントロジー側にあって実体に無い列を除くため。"""
    if path not in _columns_cache:
        try:
            _columns_cache[path] = set(spark.read.format("delta").load(path).columns)
        except Exception as e:
            print(f"  [警告] 列を読み取れません: {path}")
            print(f"         {type(e).__name__}: {e}")
            _columns_cache[path] = None
    return _columns_cache[path]


def read_ontology_model(definition_parts):
    """オントロジーの定義から、グラフ生成に必要な情報を取り出す。"""
    entity_defs, entity_by_id, entity_bindings = {}, {}, {}
    rel_defs, rel_by_id, rel_contexts = {}, {}, {}

    for p in definition_parts:
        path = p["path"]
        if path.startswith("EntityTypes/") and path.endswith("/definition.json"):
            d = unb64(p["payload"])
            entity_defs[d["name"]] = d
            entity_by_id[d["id"]] = d["name"]
        elif path.startswith("RelationshipTypes/") and path.endswith("/definition.json"):
            d = unb64(p["payload"])
            rel_defs[d["name"]] = d
            rel_by_id[d["id"]] = d["name"]

    for p in definition_parts:
        path = p["path"]
        if "/DataBindings/" in path:
            name = entity_by_id.get(path.split("/")[1])
            if name:
                entity_bindings[name] = unb64(p["payload"])
        elif "/Contextualizations/" in path:
            name = rel_by_id.get(path.split("/")[1])
            if name:
                rel_contexts[name] = unb64(p["payload"])

    return {"entities": entity_defs, "entity_by_id": entity_by_id,
            "bindings": entity_bindings, "relationships": rel_defs,
            "contexts": rel_contexts}


def build_graph_definition(model):
    """オントロジーの情報から、グラフの定義3ファイルを組み立てる。"""
    node_types, node_tables, edge_types, edge_tables = [], [], [], []
    data_sources, source_ids = [], {}
    skipped = []

    def data_source_id(ws_id, item_id, schema, table):
        """テーブル1つにつき1つのデータソースを共有する。"""
        key = (ws_id, item_id, schema, table)
        if key not in source_ids:
            source_ids[key] = stable_guid("graph-ds", ws_id, item_id,
                                          str(schema), table)
            data_sources.append({
                "id": source_ids[key],
                "name": table,
                "type": "DeltaTable",
                "properties": {
                    "path": onelake_path(ws_id, item_id, schema, table),
                    "workspaceId": ws_id,
                    "itemId": item_id,
                    "schemaName": schema,
                    "tableName": table,
                },
            })
        return source_ids[key]

    # --- ノード（エンティティ型） ---
    for name in sorted(model["entities"]):
        definition = model["entities"][name]
        binding = model["bindings"].get(name)
        if not binding:
            skipped.append((name, "データバインドがありません"))
            continue

        config = binding["dataBindingConfiguration"]
        source = config["sourceTableProperties"]
        table = source["sourceTableName"]
        schema = source.get("sourceSchema", SILVER_SCHEMA)
        ws_id, item_id = source["workspaceId"], source["itemId"]
        columns = actual_columns(onelake_path(ws_id, item_id, schema, table))

        names = {pr["id"]: pr["name"] for pr in definition["properties"]}
        types = {pr["id"]: pr["valueType"] for pr in definition["properties"]}

        properties, mappings, dropped = [], [], []
        for pb in config["propertyBindings"]:
            column, pid = pb["sourceColumnName"], pb["targetPropertyId"]
            if pid not in names:
                continue
            if columns is not None and column not in columns:
                dropped.append(column)
                continue
            properties.append({"name": names[pid],
                               "type": GRAPH_PROP_TYPES.get(types[pid], "STRING")})
            mappings.append({"sourceColumn": column, "propertyName": names[pid]})

        keys = [names[pid] for pid in definition.get("entityIdParts", [])
                if pid in names]
        column_of = {m["propertyName"]: m["sourceColumn"] for m in mappings}
        key_columns = [column_of[k] for k in keys if k in column_of]
        if not keys or len(key_columns) != len(keys):
            skipped.append((name, "キー列がシルバー層に見つかりません"))
            continue

        node_types.append({
            "name": name,
            "alias": name,
            "labels": [name],
            "primaryKeyProperties": keys,
            "properties": properties,
        })
        node_tables.append({
            "id": stable_guid("graph-node", name, table),
            "nodeType": name,
            "nodeTypeAlias": name,
            "dataSourceId": data_source_id(ws_id, item_id, schema, table),
            "dataSourceName": table,
            "schemaName": schema,
            "tableName": table,
            "keyColumns": key_columns,
            "propertyMappings": mappings,
        })
        note = f"  除外列: {dropped}" if dropped else ""
        print(f"  ノード {name:24s} <- {table:26s} "
              f"プロパティ {len(properties):2d} 件{note}")

    node_names = {n["name"] for n in node_types}

    # --- エッジ（関係型） ---
    for name in sorted(model["relationships"]):
        definition = model["relationships"][name]
        context = model["contexts"].get(name)
        if not context:
            skipped.append((name, "コンテキスト化がありません"))
            continue

        src = model["entity_by_id"].get(definition["source"]["entityTypeId"])
        dst = model["entity_by_id"].get(definition["target"]["entityTypeId"])
        if src not in node_names or dst not in node_names:
            skipped.append((name, "接続先のノードが作られていません"))
            continue

        table_info = context["dataBindingTable"]
        table = table_info["sourceTableName"]
        schema = table_info.get("sourceSchema", SILVER_SCHEMA)
        ws_id, item_id = table_info["workspaceId"], table_info["itemId"]

        src_columns = [b["sourceColumnName"]
                       for b in context.get("sourceKeyRefBindings", [])]
        dst_columns = [b["sourceColumnName"]
                       for b in context.get("targetKeyRefBindings", [])]
        if not src_columns or not dst_columns:
            skipped.append((name, "結合列が定義されていません"))
            continue

        columns = actual_columns(onelake_path(ws_id, item_id, schema, table))
        if columns is not None:
            missing = [c for c in src_columns + dst_columns if c not in columns]
            if missing:
                skipped.append((name, f"結合列がシルバー層にありません: {missing}"))
                continue

        edge_types.append({
            "alias": name,
            "labels": [name],
            "properties": [],
            "sourceNodeType": {"alias": src},
            "destinationNodeType": {"alias": dst},
        })
        edge_tables.append({
            "id": stable_guid("graph-edge", name, table),
            "edgeTypeAlias": name,
            "dataSourceName": table,
            "sourceNodeKeyColumns": src_columns,
            "destinationNodeKeyColumns": dst_columns,
        })
        data_source_id(ws_id, item_id, schema, table)
        print(f"  エッジ {name:24s}    {src} -> {dst}")

    if skipped:
        print("\n  作成しなかったもの:")
        for name, reason in skipped:
            print(f"    {name:24s} {reason}")

    return (
        {"$schema": f"{GRAPH_SCHEMA_BASE}/graphType/1.0.0/schema.json",
         "nodeTypes": node_types, "edgeTypes": edge_types},
        {"$schema": f"{GRAPH_SCHEMA_BASE}/dataSources/1.0.0/schema.json",
         "dataSources": data_sources},
        {"$schema": f"{GRAPH_SCHEMA_BASE}/graphDefinition/1.0.0/schema.json",
         "nodeTables": node_tables, "edgeTables": edge_tables},
    )


def wait_until_idle(graph_id, timeout=600):
    """進行中の取り込みが終わるのを待つ（同時実行は拒否されるため）。

    ジョブ関連だけは Core の Job Scheduler API を使う（公式のルートが
    /workspaces/{ws}/items/{itemId}/jobs/... で、アイテム種別によらず共通のため）。
    定義の取得・更新は種別ごとの /ontologies/... /graphModels/... を使う。
    """
    deadline = time.time() + timeout
    while time.time() < deadline:
        jobs = api("GET", f"/workspaces/{WORKSPACE_ID}/items/{graph_id}"
                          f"/jobs/instances").get("value", [])
        running = [j for j in jobs if j["status"] in ("NotStarted", "InProgress")]
        if not running:
            return True
        print(f"    既存の取り込みを待っています（{len(running)} 件）")
        time.sleep(20)
    return False


def refresh_graph(graph_id, timeout=REFRESH_TIMEOUT_SEC):
    """グラフへのデータ取り込みを実行し、完了まで待つ。"""
    if not wait_until_idle(graph_id):
        print("  既存の取り込みが終わらないため、中止しました。")
        return False

    r = requests.post(
        f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{graph_id}"
        f"/jobs/RefreshGraph/instances", headers=HEADERS, json={})
    if r.status_code != 202:
        print(f"  取り込みを開始できません: {r.status_code} {r.text[:300]}")
        return False
    print("  取り込みを開始しました。")

    if not WAIT_FOR_REFRESH:
        print("  完了を待たずに終了します（WAIT_FOR_REFRESH = False）。")
        return True

    deadline, last = time.time() + timeout, None
    while time.time() < deadline:
        time.sleep(15)
        jobs = api("GET", f"/workspaces/{WORKSPACE_ID}/items/{graph_id}"
                          f"/jobs/instances").get("value", [])
        if not jobs:
            continue
        job = jobs[0]
        state = (job["status"], job.get("startTimeUtc"))
        if state != last:
            last = state
            print(f"    {job['status']:12s} {job.get('startTimeUtc')}")
        if job["status"] == "Completed":
            print("  取り込みが完了しました。")
            return True
        if job["status"] == "Failed":
            reason = job.get("failureReason") or {}
            print(f"  取り込みに失敗しました: {reason.get('errorCode')}")
            print(f"    {reason.get('message', '')[:600]}")
            return False
    print("  待ち時間内に完了しませんでした。しばらくしてから状態を確認してください。")
    return False


if BUILD_GRAPH:
    print("\n" + "=" * 70)
    print("フェーズ4: グラフを構築する")
    print("=" * 70)

    graphs = list_items("GraphModel")
    graph = next((g for g in graphs
                  if g["displayName"].startswith(ONTOLOGY_NAME)), None) \
        or (graphs[0] if graphs else None)

    if graph is None:
        print("[エラー] グラフが見つかりません。")
        print("        オントロジーを作成すると自動で作られるはずのものです。")
        print("        ワークスペースにオントロジーが存在するか確認してください。")
    else:
        GRAPH_ID = graph["id"]
        print(f"グラフ: {graph['displayName']}\n")

        model = read_ontology_model(ap)
        graph_type, graph_sources, graph_binding = build_graph_definition(model)

        print(f"\n  ノード {len(graph_type['nodeTypes'])} 種類 / "
              f"エッジ {len(graph_type['edgeTypes'])} 種類 / "
              f"データソース {len(graph_sources['dataSources'])} 件")

        current = api("POST", f"/workspaces/{WORKSPACE_ID}/graphModels"
                              f"/{GRAPH_ID}/getDefinition")
        replacements = {
            "graphType.json": graph_type,
            "dataSources.json": graph_sources,
            "graphDefinition.json": graph_binding,
        }
        graph_parts = [
            {"path": p["path"], "payload": b64(replacements[p["path"]]),
             "payloadType": "InlineBase64"}
            if p["path"] in replacements else p
            for p in current["definition"]["parts"]
        ]

        api("POST", f"/workspaces/{WORKSPACE_ID}/graphModels/{GRAPH_ID}"
                    f"/updateDefinition",
            json={"definition": {"parts": graph_parts}})
        print("  グラフの定義を更新しました。\n")

        if refresh_graph(GRAPH_ID):
            print("\n" + "=" * 70)
            print("グラフの構築まで完了しました。")
            print("Fabric でグラフを開くと、ノードとエッジが表示されます。")
            print("この状態で Step 3（MCP エンドポイント）に進めます。")
            print("=" * 70)
        else:
            print("\n[注意] グラフへの取り込みが完了していません。")
            print("       このままでは MCP からのデータ検索が失敗します。")
            print("       エラー内容を確認し、シルバー層のテーブルが")
            print("       マネージドテーブルとして作成されているか確認してください。")